# RL Market Maker Training + Evaluation

This notebook trains PPO and SAC agents on `MarketMakerEnv` and compares them against a simple inventory-skewed heuristic policy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lumina_lob.rl import (
    MarketMakerEnv,
    SimpleMarketMakerPolicy,
    evaluate_agent,
    evaluate_heuristic_policy,
    make_env,
    summarize_results,
    train_ppo,
    train_sac,
)

plt.style.use('seaborn-v0_8-whitegrid')

## Environment

In [ ]:
env_factory = make_env(seed=42)
env = env_factory()
env.reset(seed=42)
print('Observation shape:', env.observation_space.shape)
print('Action shape:', env.action_space.shape)

## Train PPO baseline

In [ ]:
ppo_model = train_ppo(
    env,
    total_timesteps=5_000,
    verbose=1,
    device='cpu',
)

## Train SAC comparison

In [ ]:
sac_model = train_sac(
    env_factory(),
    total_timesteps=5_000,
    verbose=1,
    device='cpu',
)

## Evaluate trained agents

In [ ]:
ppo_mean, ppo_std = evaluate_agent(ppo_model, env_factory(), n_eval_episodes=5)
sac_mean, sac_std = evaluate_agent(sac_model, env_factory(), n_eval_episodes=5)

print(f'PPO mean reward: {ppo_mean:.3f} +/- {ppo_std:.3f}')
print(f'SAC mean reward: {sac_mean:.3f} +/- {sac_std:.3f}')

## Evaluate heuristic inventory-skewed policy

In [ ]:
heuristic_results = evaluate_heuristic_policy(
    env_factory,
    SimpleMarketMakerPolicy(),
    n_episodes=5,
)
heuristic_summary = summarize_results(heuristic_results)
heuristic_summary

## Comparison plot

In [ ]:
summary = pd.DataFrame(
    [
        {'Agent': 'PPO', 'Mean reward': ppo_mean, 'Std': ppo_std},
        {'Agent': 'SAC', 'Mean reward': sac_mean, 'Std': sac_std},
        {
            'Agent': 'Heuristic',
            'Mean reward': heuristic_summary['mean_reward'],
            'Std': np.std([r.total_reward for r in heuristic_results]),
        },
    ]
)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summary))
ax.bar(x, summary['Mean reward'], yerr=summary['Std'], capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(summary['Agent'])
ax.set_ylabel('Episode reward')
ax.set_title('RL vs Heuristic Market-Maker Performance')
plt.show()